# Itô Calculus --- Numerical Exploration

This notebook accompanies node 34. We use **only the Python standard library**.

We numerically illustrate:
1. The Itô integral defined as a **left-endpoint** Riemann--Stieltjes sum.
2. Why right-endpoint sums give a different (biased) answer.
3. Itôs lemma applied to \(f(x) = x^2\):
\[ d(W_t^2) = 2 W_t\,dW_t + dt. \]
4. Quadratic variation: \(\sum (\Delta W_i)^2 \to t\).


In [ ]:
import math, random
random.seed(0)

def brownian_path(n, dt, rng):
    sd = math.sqrt(dt)
    inc = [rng.gauss(0.0, sd) for _ in range(n)]
    path = [0.0]
    for dW in inc:
        path.append(path[-1] + dW)
    return path, inc

N, T = 1000, 1.0
dt = T / N
rng = random.Random(0)
path, inc = brownian_path(N, dt, rng)
print(f"W_T = {path[-1]:+.4f},  number of increments = {len(inc)}")

## 1. Quadratic variation

For Brownian motion, \([W]_T = T\) almost surely. We verify by summing squared increments.

In [ ]:
qv = sum(d*d for d in inc)
print(f"sum (dW)^2 = {qv:.4f}   (should approach T = {T})")

## 2. Left- vs right-endpoint sums

Itôs choice is the **left** endpoint. The right-endpoint sum is biased by \(+T\).

In [ ]:
def left_sum(path, inc):
    return sum(path[i]*inc[i] for i in range(len(inc)))
def right_sum(path, inc):
    return sum(path[i+1]*inc[i] for i in range(len(inc)))

M = 1000
rng = random.Random(1)
Ls, Rs = [], []
for _ in range(M):
    p, c = brownian_path(N, dt, rng)
    Ls.append(left_sum(p, c))
    Rs.append(right_sum(p, c))

mL = sum(Ls)/M; mR = sum(Rs)/M
print(f"E[left ]  approx {mL:+.4f}   (Itô; should be ~ -T/2 = -0.5)")
print(f"E[right]  approx {mR:+.4f}   (biased; should be ~ +T/2 = +0.5)")
print(f"E[right - left] approx {mR - mL:+.4f}   (should be ~ T = 1.0; this is Itôs correction)")

## 3. Itôs lemma check on a single path

\[ W_T^2 - T = 2 \int_0^T W_s\,dW_s. \]
We compare the closed form (left side) to twice the left-endpoint sum (right side) on one path.

In [ ]:
rng = random.Random(42)
p, c = brownian_path(N, dt, rng)
W_T = p[-1]
L = left_sum(p, c)
print(f"W_T^2 - T       = {W_T*W_T - T:+.4f}")
print(f"2 * left_sum    = {2*L:+.4f}")
print(f"discrepancy     = {(W_T*W_T - T) - 2*L:+.4f}   (Euler error, -> 0 as N -> inf)")

## 4. Convergence as \(N\) grows

The pathwise discrepancy between \((W_T^2 - T)/2\) and the left-sum estimator should shrink like \(1/\sqrt{N}\).

In [ ]:
rng = random.Random(7)
for N_try in [50, 200, 1000, 5000]:
    dt_try = T / N_try
    errs = []
    for _ in range(200):
        p, c = brownian_path(N_try, dt_try, rng)
        L = left_sum(p, c)
        ito = 0.5*(p[-1]**2 - T)
        errs.append(L - ito)
    rms = math.sqrt(sum(e*e for e in errs)/len(errs))
    print(f"N = {N_try:5d}   RMS pathwise error = {rms:.4f}")

## 5. Takeaways

- The Itô integral is the **left-endpoint** \(L^2\) limit; this preserves adaptedness and the martingale property.
- The right-endpoint convention disagrees by exactly \(T\) in expectation --- this gap **is** Itôs correction.
- Itôs lemma \(d(W_t^2) = 2 W\,dW + dt\) is verified pathwise to within Euler error.
- Quadratic variation \(\sum (\Delta W_i)^2 \to T\) is the structural fact that forces the second-order term.
